---
# Phase 8: T2U Knowledge Distillation (Audio Translation Recovery)

**Goal:** Recover audio-translation quality lost from T2U pruning (Phase 6) by distilling
from the full teacher (`facebook/seamless-m4t-v2-large`) into the T2U sub-model of
`phase7_dora_merged_v1`. All Phase 7 DoRA-recovered components are **frozen**; only the
pruned T2U encoder + decoder are updated.

**Loss:** KL-divergence on T2U decoder logits (soft-label KD, temperature T=2) +
optional cross-entropy on hard unit labels.

| Component | Status |
|-----------|--------|
| `speech_encoder` | ❄️ Frozen |
| `text_decoder`   | ❄️ Frozen |
| `lm_head` / `shared` | ❄️ Frozen |
| `vocoder`        | ❄️ Frozen |
| `t2u_model`      | 🔥 **Trained via KD** |

## Phase 8 — Cell 1: Load Phase 7 Student Model & Freeze Non-T2U Components

In [ ]:
# ── Phase 8 Cell 1: Load phase7_dora_merged_v1 and freeze everything except T2U ──
import gc as _gc

# Free any model from memory that may still be resident from Phase 7
for _var in ['model_p7', 'model_p7_merged']:
    if _var in dir():
        del globals()[_var]
_gc.collect()
torch.cuda.empty_cache()

print('[P8] Loading phase7_dora_merged_v1 as student...')
model_p8_student, processor = load_model_from_drive('phase7_dora_merged_v1')
model_p8_student.train()

# ── Freeze everything EXCEPT t2u_model ────────────────────────────────────────
frozen_parts   = ['speech_encoder', 'text_decoder', 'lm_head', 'shared', 'vocoder']
trainable_t2u  = []

for name, param in model_p8_student.named_parameters():
    top_module = name.split('.')[0]
    if top_module in frozen_parts:
        param.requires_grad_(False)
    elif top_module == 't2u_model':
        param.requires_grad_(True)
        trainable_t2u.append(param)
    else:
        # Safety: freeze anything unexpected
        param.requires_grad_(False)

total_params    = sum(p.numel() for p in model_p8_student.parameters())
trainable_count = sum(p.numel() for p in trainable_t2u)
print(f'[P8] Total params      : {total_params/1e6:.1f}M')
print(f'[P8] Trainable (T2U)   : {trainable_count/1e6:.1f}M  ({trainable_count/total_params*100:.1f}%)')
gpu_mem()

## Phase 8 — Cell 2: Load Teacher Model for KD

In [ ]:
# ── Phase 8 Cell 2: Load full teacher model ────────────────────────────────────
# Uses the existing load_base_model() from Cell 20.
# Teacher is always in eval() mode with no_grad — never updated.

print('[P8] Loading teacher (facebook/seamless-m4t-v2-large)...')
model_teacher, _proc_teacher = load_base_model()   # load_base_model() defined in setup cells
model_teacher.eval()
for p in model_teacher.parameters():
    p.requires_grad_(False)

print(f'[P8] Teacher params : {count_params(model_teacher):.1f}M')
print(f'[P8] Student params : {count_params(model_p8_student):.1f}M')
gpu_mem()

## Phase 8 — Cell 3: T2U KD Loss & Training Utilities

In [ ]:
# ── Phase 8 Cell 3: KD loss helpers ───────────────────────────────────────────
import torch.nn.functional as F
import logging

# ── KD hyper-parameters ───────────────────────────────────────────────────────
KD_TEMPERATURE   = 2.0     # Soft-label temperature for KL divergence
KD_ALPHA         = 0.7     # Weight for soft KD loss  (1-ALPHA → hard CE loss)
KD_MAX_STEPS     = 500     # Optimiser steps (increase for better recovery)
KD_BATCH_SIZE    = 2       # Speech samples per forward pass (VRAM-limited)
KD_GRAD_ACCUM    = 4       # Effective batch = KD_BATCH_SIZE * KD_GRAD_ACCUM
KD_LR            = 3e-5
KD_GRAD_CLIP     = 1.0
KD_LOG_EVERY     = 25
KD_SAVE_EVERY    = 100


def _get_t2u_inputs_via_teacher(teacher, student, wav_batch, tgt_lang='ben'):
    """
    Run the shared speech-encoder + text-decoder on the TEACHER to get
    the unit-label sequence that feeds the T2U model, then run the T2U
    encoder/decoder on BOTH teacher and student.

    Returns:
        t2u_logits_student  : (B, seq, vocab)  — from student T2U
        t2u_logits_teacher  : (B, seq, vocab)  — from teacher T2U  (no_grad)
        unit_labels         : (B, seq)          — hard unit label ids
    """
    device = next(student.parameters()).device
    inputs = processor(audio=wav_batch, sampling_rate=16000,
                       return_tensors='pt', padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ── 1. Get speech encoder output (shared between teacher and student frames)
    #    We run the full forward on teacher with labels to get unit-token targets.
    with torch.no_grad():
        teacher_out = teacher(
            **inputs,
            tgt_lang=tgt_lang,
            return_dict=True,
            output_hidden_states=False,
        )

    # ── 2. Soft labels from teacher T2U logits ─────────────────────────────────
    #    SeamlessM4Tv2 .generate() returns waveform; for logits we need a
    #    custom pass that exposes the t2u_model logits.
    #    Strategy: hook the teacher t2u decoder output_projection.
    teacher_t2u_logits = []
    student_t2u_logits = []

    def _hook_teacher(module, inp, out):
        if isinstance(out, torch.Tensor):
            teacher_t2u_logits.append(out.detach().float())

    def _hook_student(module, inp, out):
        if isinstance(out, torch.Tensor):
            student_t2u_logits.append(out.float())

    # Identify output projection in T2U decoder
    def _find_output_proj(mdl):
        t2u = mdl.t2u_model
        # Try common attribute names
        for attr in ['output_projection', 'lm_head', 'embed_tokens']:
            if hasattr(t2u, attr):
                return getattr(t2u, attr)
        # Walk children to find a Linear with large out_features
        for name, mod in t2u.named_modules():
            if isinstance(mod, torch.nn.Linear) and mod.out_features > 500:
                return mod
        return None

    t_proj = _find_output_proj(teacher)
    s_proj = _find_output_proj(student)

    if t_proj is None or s_proj is None:
        raise RuntimeError('[P8] Could not locate T2U output projection layer.')

    h_t = t_proj.register_forward_hook(_hook_teacher)
    h_s = s_proj.register_forward_hook(_hook_student)

    try:
        with torch.no_grad():
            teacher.generate(**inputs, tgt_lang=tgt_lang)
        student.generate(**inputs, tgt_lang=tgt_lang)
    finally:
        h_t.remove()
        h_s.remove()

    if not teacher_t2u_logits or not student_t2u_logits:
        raise RuntimeError('[P8] Hook did not capture T2U logits.')

    # Stack calls: hooks fire once per beam step → take the first (greedy step)
    t_logits = teacher_t2u_logits[0]   # (B, V)
    s_logits = student_t2u_logits[0]   # (B, V)

    # Hard labels from teacher argmax
    unit_labels = t_logits.argmax(dim=-1)  # (B,)

    return s_logits, t_logits, unit_labels


def compute_kd_loss(s_logits, t_logits, hard_labels, temperature=KD_TEMPERATURE, alpha=KD_ALPHA):
    """
    KD loss = alpha * KL(soft_teacher || soft_student) + (1-alpha) * CE(student, hard)
    """
    # Soft KD loss (KL divergence)
    T = temperature
    s_soft = F.log_softmax(s_logits / T, dim=-1)
    t_soft = F.softmax(t_logits / T, dim=-1)
    kl_loss = F.kl_div(s_soft, t_soft, reduction='batchmean') * (T ** 2)

    # Hard CE loss
    ce_loss = F.cross_entropy(s_logits, hard_labels.to(s_logits.device))

    return alpha * kl_loss + (1.0 - alpha) * ce_loss, kl_loss.item(), ce_loss.item()


print('[P8] KD loss helpers ready.')
print(f'     Temperature={KD_TEMPERATURE}  Alpha={KD_ALPHA}  MaxSteps={KD_MAX_STEPS}')

## Phase 8 — Cell 4: Optimiser Setup

In [ ]:
# ── Phase 8 Cell 4: Optimiser + scheduler (T2U params only) ───────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import random

kd_optimizer = AdamW(
    trainable_t2u,
    lr=KD_LR,
    betas=(0.9, 0.999),
    weight_decay=0.01,
)
kd_scheduler = CosineAnnealingLR(kd_optimizer, T_max=KD_MAX_STEPS, eta_min=KD_LR / 20)

# ── Resume from checkpoint if available ───────────────────────────────────────
kd_ckpt = load_latest_checkpoint('phase8_kd')
kd_start_step = 0
kd_loss_log   = []
kd_kl_log     = []
kd_ce_log     = []

if kd_ckpt and kd_ckpt.get('step', 0) > 0:
    kd_start_step = kd_ckpt['step']
    kd_loss_log   = kd_ckpt.get('loss_log', [])
    kd_kl_log     = kd_ckpt.get('kl_log', [])
    kd_ce_log     = kd_ckpt.get('ce_log', [])
    if kd_ckpt.get('optimizer_state'):
        kd_optimizer.load_state_dict(kd_ckpt['optimizer_state'])
    if kd_ckpt.get('scheduler_state'):
        kd_scheduler.load_state_dict(kd_ckpt['scheduler_state'])
    print(f'[P8] Resuming KD from step {kd_start_step}')
else:
    print('[P8] Starting KD from scratch.')

print(f'[P8] Optimiser: AdamW  LR={KD_LR}  EffectiveBatch={KD_BATCH_SIZE * KD_GRAD_ACCUM}')

## Phase 8 — Cell 5: T2U KD Training Loop

In [ ]:
# ── Phase 8 Cell 5: KD training loop ──────────────────────────────────────────
# Suppress verbose HF layer warnings during training
_hf_logger = logging.getLogger(
    'transformers.models.seamless_m4t_v2.modeling_seamless_m4t_v2')
_prev_level = _hf_logger.level
_hf_logger.setLevel(logging.ERROR)

try:
    model_p8_student.train()
    model_teacher.eval()

    optim_steps     = kd_start_step
    micro_step      = 0
    consecutive_err = 0
    kd_optimizer.zero_grad()
    t0 = time.time()

    while optim_steps < KD_MAX_STEPS:
        batch = random.sample(ft_samples, min(KD_BATCH_SIZE, len(ft_samples)))
        wav_batch = [s['wav'] for s in batch]

        try:
            s_logits, t_logits, unit_labels = _get_t2u_inputs_via_teacher(
                model_teacher, model_p8_student, wav_batch, tgt_lang=TARGET_LANG
            )
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                loss, kl_val, ce_val = compute_kd_loss(
                    s_logits, t_logits, unit_labels
                )
                loss = loss / KD_GRAD_ACCUM
            loss.backward()
            consecutive_err = 0

        except Exception as e:
            consecutive_err += 1
            print(f'  [ERR] Step {optim_steps}: {e}')
            if consecutive_err > 5:
                print('[P8] CRITICAL: Too many consecutive errors. Stopping.')
                break
            kd_optimizer.zero_grad()
            continue

        kd_loss_log.append(loss.item() * KD_GRAD_ACCUM)
        kd_kl_log.append(kl_val)
        kd_ce_log.append(ce_val)

        if (micro_step + 1) % KD_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable_t2u, KD_GRAD_CLIP)
            kd_optimizer.step()
            kd_scheduler.step()
            kd_optimizer.zero_grad()
            optim_steps += 1

            if optim_steps % KD_LOG_EVERY == 0:
                window = kd_loss_log[-KD_LOG_EVERY:]
                avg_loss = sum(window) / len(window)
                avg_kl   = sum(kd_kl_log[-KD_LOG_EVERY:]) / KD_LOG_EVERY
                avg_ce   = sum(kd_ce_log[-KD_LOG_EVERY:]) / KD_LOG_EVERY
                elapsed  = time.time() - t0
                lr_now   = kd_scheduler.get_last_lr()[0]
                print(f'[P8] Step {optim_steps:>4}/{KD_MAX_STEPS}'
                      f'  loss={avg_loss:.4f}  KL={avg_kl:.4f}  CE={avg_ce:.4f}'
                      f'  lr={lr_now:.2e}  {elapsed/60:.1f}min')

            if optim_steps % KD_SAVE_EVERY == 0:
                save_checkpoint(
                    dict(step=optim_steps,
                         loss_log=kd_loss_log, kl_log=kd_kl_log, ce_log=kd_ce_log,
                         optimizer_state=kd_optimizer.state_dict(),
                         scheduler_state=kd_scheduler.state_dict()),
                    name='phase8_kd', step=optim_steps)

        micro_step += 1

    print(f'\n[P8] KD complete. Final step: {optim_steps}'
          f'  Time: {(time.time()-t0)/60:.1f} min')

    # Final checkpoint save
    save_checkpoint(
        dict(step=optim_steps,
             loss_log=kd_loss_log, kl_log=kd_kl_log, ce_log=kd_ce_log,
             optimizer_state=kd_optimizer.state_dict(),
             scheduler_state=kd_scheduler.state_dict()),
        name='phase8_kd', step=optim_steps)

finally:
    _hf_logger.setLevel(_prev_level)

## Phase 8 — Cell 6: Plot KD Training Curves

In [ ]:
# ── Phase 8 Cell 6: Training loss plot ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

def _smooth(vals, w=10):
    if len(vals) < w:
        return vals
    return [float(np.mean(vals[max(0, i-w):i+1])) for i in range(len(vals))]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Phase 8 — T2U Knowledge Distillation Training', fontsize=13, fontweight='bold')

for ax, log, title, color in zip(
    axes,
    [kd_loss_log, kd_kl_log, kd_ce_log],
    ['Total KD Loss', 'KL Divergence (soft)', 'Cross-Entropy (hard)'],
    ['#E91E63', '#2196F3', '#FF9800'],
):
    xs = list(range(1, len(log) + 1))
    ax.plot(xs, log, alpha=0.25, color=color, linewidth=0.8)
    ax.plot(xs, _smooth(log, 15), color=color, linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Micro-step')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, 'phase8_kd_training_curves.png')
plt.show()
print('[P8] Training curve saved.')

## Phase 8 — Cell 7: Save phase8_kd Model to Drive

In [ ]:
# ── Phase 8 Cell 7: Save KD-trained student model ─────────────────────────────
model_p8_student.eval()
sync_model_config(model_p8_student)

save_model_to_drive(model_p8_student, processor, 'phase8_kd')
print_model_breakdown(model_p8_student, 'After Phase 8: T2U KD')

---
## Phase 8 — Benchmark: 4-Model Comparison

Evaluate **ASR-BLEU**, **ASR-ChrF**, **Text-BLEU**, **Text-ChrF** for:

| # | Model | Description |
|---|-------|-------------|
| 1 | `base_model` (teacher) | `facebook/seamless-m4t-v2-large` |
| 2 | `phase6_t2u_iter_pruned` | After T2U layer pruning |
| 3 | `phase7_dora_merged_v1` | After DoRA text recovery |
| 4 | `phase8_kd` | After T2U KD audio recovery (**final student**) |

### Benchmark Cell 1: Extended Benchmark Function (ASR + Text metrics)

In [ ]:
# ── Phase 8 Benchmark Cell 1: run_benchmark_full() ────────────────────────────
# Extends the existing run_benchmark() to also compute ASR-BLEU + ASR-ChrF
# by running the synthesised audio through the MMS Bengali ASR model.

def run_benchmark_full(mdl, samples, label='model', tgt_lang='ben', save_n=2):
    """
    Full benchmark computing:
      - text-BLEU  : decoded text vs Bengali reference
      - text-ChrF  : decoded text vs Bengali reference
      - ASR-BLEU   : MMS-ASR(synthesised audio) vs Bengali reference
      - ASR-ChrF   : MMS-ASR(synthesised audio) vs Bengali reference

    Returns (results_list, summary_dict)
    """
    print(f'\n{"="*60}\n  BENCHMARK (full): {label}\n  Samples: {len(samples)}  Target: {tgt_lang}\n{"="*60}\n')
    gpu_mem()

    results = []
    for i, s in enumerate(samples):
        try:
            dur = len(s['wav']) / 16000
            t0  = time.time()

            # Full S2ST (text + audio)
            pred_text, out_wav = run_s2st(mdl, s['wav'], tgt_lang=tgt_lang)
            elapsed = time.time() - t0
            rtf     = elapsed / max(dur, 1e-6)

            # Text metrics
            t_bleu = compute_bleu(pred_text, s['ref'])
            t_chrf = compute_chrf(pred_text, s['ref'])

            # ASR metrics — transcribe synthesised audio then score
            asr_hyp, asr_bleu = compute_asr_bleu(out_wav, s['ref'])
            _,       asr_chrf = compute_asr_chrf(out_wav, s['ref'])

            print(f'  [{i+1:>2}/{len(samples)}] '
                  f'T-BLEU={t_bleu:5.1f}  T-ChrF={t_chrf:5.1f}  '
                  f'ASR-BLEU={asr_bleu:5.1f}  ASR-ChrF={asr_chrf:5.1f}  '
                  f'RTF={rtf:.3f}  id={s["id"]}')
            print(f'         text: {pred_text[:70]}')
            print(f'          asr: {asr_hyp[:70]}')

            # Optionally save audio clips
            if save_n > 0 and i < save_n:
                save_audio(s['wav'],  16000,                       f'{label}_s{i+1}_in.wav')
                save_audio(out_wav,   mdl.config.sampling_rate,    f'{label}_s{i+1}_out.wav')
                play(out_wav, mdl.config.sampling_rate, f'{label}_s{i+1}_out.wav')

            results.append(dict(
                id=s['id'],
                text_bleu=t_bleu, text_chrf=t_chrf,
                asr_bleu=asr_bleu, asr_chrf=asr_chrf,
                rtf=rtf, pred=pred_text, asr=asr_hyp, ref=s['ref']
            ))

        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  [{i+1:>2}/{len(samples)}] ERROR: {e}')
            results.append(dict(
                id=s['id'],
                text_bleu=0, text_chrf=0,
                asr_bleu=0, asr_chrf=0,
                rtf=float('nan'), pred='', asr='', ref=s.get('ref', '')
            ))

    valid = [r for r in results if not math.isnan(r['rtf'])]
    def _avg(key): return float(np.mean([r[key] for r in valid])) if valid else 0.0

    summary = dict(
        label=label,
        n=len(valid),
        avg_text_bleu  = _avg('text_bleu'),
        avg_text_chrf  = _avg('text_chrf'),
        avg_asr_bleu   = _avg('asr_bleu'),
        avg_asr_chrf   = _avg('asr_chrf'),
        # Keep legacy keys for backward-compat with plot_phase_comparison()
        avg_bleu       = _avg('text_bleu'),
        avg_chrf       = _avg('text_chrf'),
        avg_rtf        = _avg('rtf'),
        params_M       = count_params(mdl),
    )

    print(f'\n  [{label}] Summary:')
    print(f'    Text-BLEU  = {summary["avg_text_bleu"]:.2f}')
    print(f'    Text-ChrF  = {summary["avg_text_chrf"]:.2f}')
    print(f'    ASR-BLEU   = {summary["avg_asr_bleu"]:.2f}')
    print(f'    ASR-ChrF   = {summary["avg_asr_chrf"]:.2f}')
    print(f'    RTF        = {summary["avg_rtf"]:.4f}')
    print(f'    Params     = {summary["params_M"]:.1f}M\n')

    return results, summary

print('[P8] run_benchmark_full() ready.')

### Benchmark Cell 2: Run Benchmark on All 4 Models

In [ ]:
# ── Phase 8 Benchmark Cell 2: Evaluate all four models ─────────────────────────
# Each model is loaded, benchmarked, then immediately freed from VRAM.
# The teacher (model_teacher) may already be in memory; the rest are loaded fresh.

p8_bench_summaries = {}   # label -> summary dict

# ── 1. Teacher (base model) ────────────────────────────────────────────────────
print('\n' + '='*60)
print('[P8 BENCH] 1/4 — base_model (teacher)')
print('='*60)
model_teacher.eval()
_, summ = run_benchmark_full(model_teacher, eval_samples,
                              label='base_model', tgt_lang=TARGET_LANG, save_n=0)
p8_bench_summaries['base_model'] = summ
store_summary({**summ, 'label': 'P0_Baseline_Full'})

# ── 2. phase6_t2u_iter_pruned ─────────────────────────────────────────────────
print('\n' + '='*60)
print('[P8 BENCH] 2/4 — phase6_t2u_iter_pruned')
print('='*60)
_gc.collect(); torch.cuda.empty_cache()
model_p6, _ = load_model_from_drive('phase6_t2u_iter_pruned')
model_p6.eval()
_, summ = run_benchmark_full(model_p6, eval_samples,
                              label='phase6_t2u_iter_pruned', tgt_lang=TARGET_LANG, save_n=0)
p8_bench_summaries['phase6_t2u_iter_pruned'] = summ
store_summary({**summ, 'label': 'P6_T2UIter_Full'})
del model_p6; _gc.collect(); torch.cuda.empty_cache()

# ── 3. phase7_dora_merged_v1 ──────────────────────────────────────────────────
print('\n' + '='*60)
print('[P8 BENCH] 3/4 — phase7_dora_merged_v1')
print('='*60)
model_p7_ref, _ = load_model_from_drive('phase7_dora_merged_v1')
model_p7_ref.eval()
_, summ = run_benchmark_full(model_p7_ref, eval_samples,
                              label='phase7_dora_merged_v1', tgt_lang=TARGET_LANG, save_n=0)
p8_bench_summaries['phase7_dora_merged_v1'] = summ
store_summary({**summ, 'label': 'P7_DoRA_Full'})
del model_p7_ref; _gc.collect(); torch.cuda.empty_cache()

# ── 4. phase8_kd (final student) ──────────────────────────────────────────────
print('\n' + '='*60)
print('[P8 BENCH] 4/4 — phase8_kd  (final student)')
print('='*60)
model_p8_student.eval()
_, summ = run_benchmark_full(model_p8_student, eval_samples,
                              label='phase8_kd', tgt_lang=TARGET_LANG, save_n=2)
p8_bench_summaries['phase8_kd'] = summ
store_summary({**summ, 'label': 'P8_KD_Final'})

print('\n[P8] All benchmarks complete.')

### Benchmark Cell 3: Comparison Plot — ASR-BLEU, ASR-ChrF, Text-BLEU, Text-ChrF

In [ ]:
# ── Phase 8 Benchmark Cell 3: 4-metric comparison figure ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Ordered display names & model keys ────────────────────────────────────────
BENCH_ORDER = [
    ('base_model',             'Teacher\n(baseline)'),
    ('phase6_t2u_iter_pruned', 'P6 T2U\nPruned'),
    ('phase7_dora_merged_v1',  'P7 DoRA\nMerged'),
    ('phase8_kd',              'P8 KD\n(final)'),
]

PALETTE = {
    'base_model':             '#607D8B',   # blue-grey  (reference)
    'phase6_t2u_iter_pruned': '#FF7043',   # deep-orange (quality dip)
    'phase7_dora_merged_v1':  '#42A5F5',   # blue        (partial recovery)
    'phase8_kd':              '#66BB6A',   # green       (final student)
}

METRICS = [
    ('avg_text_bleu', 'Text-BLEU',  '↑ higher is better'),
    ('avg_text_chrf', 'Text-ChrF',  '↑ higher is better'),
    ('avg_asr_bleu',  'ASR-BLEU',   '↑ higher is better'),
    ('avg_asr_chrf',  'ASR-ChrF',   '↑ higher is better'),
]

x      = np.arange(len(BENCH_ORDER))
width  = 0.18
labels = [dn for _, dn in BENCH_ORDER]
keys   = [k  for k, _ in BENCH_ORDER]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    'Phase 8 — 4-Model Quality Comparison\n'
    '(Teacher  ·  P6 Pruned  ·  P7 DoRA  ·  P8 KD Final)',
    fontsize=14, fontweight='bold', y=1.01
)

for ax, (metric_key, metric_title, metric_note) in zip(axes.flat, METRICS):
    vals   = [p8_bench_summaries[k].get(metric_key, 0.0) for k in keys]
    colors = [PALETTE[k] for k in keys]
    bars   = ax.bar(x, vals, width=0.55, color=colors,
                    edgecolor='white', linewidth=1.2, alpha=0.92)

    # Value labels on top of bars
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{val:.1f}',
            ha='center', va='bottom',
            fontsize=10, fontweight='bold'
        )

    # Reference line from teacher
    teacher_val = p8_bench_summaries['base_model'].get(metric_key, 0.0)
    ax.axhline(teacher_val, color='#607D8B', linestyle='--',
               linewidth=1.5, alpha=0.7, label=f'Teacher: {teacher_val:.1f}')

    ax.set_title(f'{metric_title}  ({metric_note})', fontweight='bold', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Score', fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim(0, max(vals + [teacher_val]) * 1.18 + 1)
    ax.legend(fontsize=9)

# ── Colour legend ──────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=PALETTE[k], label=dn.replace('\n', ' '))
    for k, dn in BENCH_ORDER
]
fig.legend(
    handles=legend_patches,
    loc='lower center',
    ncol=4,
    fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, -0.04),
)

plt.tight_layout()
save_figure(fig, 'phase8_4model_comparison.png')
plt.show()
print('[P8] Comparison figure saved → phase8_4model_comparison.png')

### Benchmark Cell 4: Radar / Spider Chart — Full Quality Profile

In [ ]:
# ── Phase 8 Benchmark Cell 4: Radar chart — quality profile per model ─────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

RADAR_METRICS = [
    ('avg_text_bleu',  'Text\nBLEU'),
    ('avg_text_chrf',  'Text\nChrF'),
    ('avg_asr_bleu',   'ASR\nBLEU'),
    ('avg_asr_chrf',   'ASR\nChrF'),
]

# Normalise each metric to [0, 1] using teacher as 100 %
teacher_vals = np.array([
    p8_bench_summaries['base_model'].get(mk, 1e-6) for mk, _ in RADAR_METRICS
])
teacher_vals = np.where(teacher_vals == 0, 1e-6, teacher_vals)  # avoid /0

N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model_key, disp_name in [
    ('base_model',             'Teacher (baseline)'),
    ('phase6_t2u_iter_pruned', 'P6 T2U Pruned'),
    ('phase7_dora_merged_v1',  'P7 DoRA Merged'),
    ('phase8_kd',              'P8 KD Final'),
]:
    raw = np.array([
        p8_bench_summaries[model_key].get(mk, 0.0) for mk, _ in RADAR_METRICS
    ])
    normed = (raw / teacher_vals * 100).tolist()
    normed += normed[:1]
    ax.plot(angles, normed, linewidth=2, label=disp_name,
            color=PALETTE[model_key])
    ax.fill(angles, normed, alpha=0.10, color=PALETTE[model_key])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([mn for _, mn in RADAR_METRICS], fontsize=11)
ax.set_rlabel_position(0)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='grey')
ax.set_ylim(0, 115)
ax.set_title('Quality Profile (% of teacher)', fontsize=13,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
ax.grid(alpha=0.35)

plt.tight_layout()
save_figure(fig, 'phase8_radar_comparison.png')
plt.show()
print('[P8] Radar chart saved → phase8_radar_comparison.png')

### Benchmark Cell 5: Print Numeric Summary Table

In [ ]:
# ── Phase 8 Benchmark Cell 5: Numeric summary table ───────────────────────────
import pandas as pd

rows = []
for k, disp in [
    ('base_model',             'Teacher (base)'),
    ('phase6_t2u_iter_pruned', 'P6 T2U Pruned'),
    ('phase7_dora_merged_v1',  'P7 DoRA Merged'),
    ('phase8_kd',              'P8 KD Final'),
]:
    s = p8_bench_summaries[k]
    rows.append({
        'Model':         disp,
        'Params (M)':    f"{s['params_M']:.1f}",
        'Text-BLEU':     f"{s.get('avg_text_bleu', 0):.2f}",
        'Text-ChrF':     f"{s.get('avg_text_chrf', 0):.2f}",
        'ASR-BLEU':      f"{s.get('avg_asr_bleu',  0):.2f}",
        'ASR-ChrF':      f"{s.get('avg_asr_chrf',  0):.2f}",
        'RTF':           f"{s.get('avg_rtf', 0):.4f}",
    })

df_results = pd.DataFrame(rows)

print('\n' + '='*80)
print('  PHASE 8 — 4-MODEL BENCHMARK SUMMARY')
print('='*80)
print(df_results.to_string(index=False))
print('='*80)

# Save as CSV to drive
csv_path = f'{FIG_DIR}/phase8_benchmark_summary.csv'
df_results.to_csv(csv_path, index=False)
if ON_KAGGLE:
    _rclone_push(csv_path, 'figures')
print(f'\n[P8] CSV saved → {csv_path}')
print('\n[P8] Phase 8 complete. Next: Phase 9 — Full benchmark + GDrive upload.')